In [7]:
require(data.table)
require(tidyverse)
require(phyloseq)
require(ggplot2)
require(RColorBrewer)
require(metacoder)
require(vegan)
require(DESeq2)

## phyloseq cleanup

In [12]:
ps<-readRDS(file = "/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/BEL_16S_ITS2/BEL_16S_outputs/ps_16S.rds")
#removing any taxa that don't show up in any samples to speed up the process
ps <- prune_taxa(taxa_sums(ps) > 0, ps)
ps

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 193447 taxa and 470 samples ]
sample_data() Sample Data:       [ 470 samples by 9 sample variables ]
tax_table()   Taxonomy Table:    [ 193447 taxa by 6 taxonomic ranks ]

In [13]:
#normalizing ps by converting rawcounts into relative abundances
#so samples with more reads wont be over represented
#using ps bc only to the count data (OTU table), while preserving the rest of the object
ps_norm = transform_sample_counts(ps, function(x) 1E6 * x / sum(x))

In [14]:
#isolate just bacteria
ps_norm_bac=subset_taxa(ps_norm, Kingdom=="Bacteria")
#remove chloroplast order
ps_norm_nochlo=subset_taxa(ps_norm_bac, Order!="Chloroplast")
#remove mitochondria family
ps_norm_nomit=subset_taxa(ps_norm_nochlo, Family!="Mitochondria")
ps_norm_nomit

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 64055 taxa and 470 samples ]
sample_data() Sample Data:       [ 470 samples by 9 sample variables ]
tax_table()   Taxonomy Table:    [ 64055 taxa by 6 taxonomic ranks ]

### convert the sample_data() within a phyloseq object to a vegan compatible data object

In [17]:
pssd2veg <- function(ps_norm_nomit) {
  sd_nomit <- sample_data(ps_norm_nomit)
  return(as(sd_nomit,"data.frame"))
}
#using phyloseq nmds plot no chloroplast
sample_nomit <- pssd2veg(ps_norm_nomit)

sample_nomit <- as.data.frame(sample_data(ps_norm_nomit))
#save sammple names as a column so tidy doesn't get rid of it during filtering
sample_nomit$SampleID <- rownames(sample_nomit)

## new repeated colonies only dataframe

In [ ]:
# Return names which have more than one row of data
# Now filter
colonies <- sample_nomit %>%
  group_by(colony) %>%
  filter(n() != 1) %>%
  ungroup()

#how many colonies are represented?
nrow(sample_nomit)
nrow(colonies)

In [ ]:
# Restore rownames
rownames(colonies) <- colonies$SampleID
#check
head(rownames(colonies))
NROW(sample_names(ps_norm_nomit))

In [ ]:
colonies <- as.data.frame(colonies)
class(colonies)

In [ ]:
#create a new phyloseq object with colonies dataframe
col_clean <- phyloseq::sample_data(colonies)
sample_data(ps_norm_nomit) <- col_clean
head(sample_data(ps_norm_nomit))

# How are the microbiomes of individual coral colonies changing overtime?
- do they recruit fewer bacteria?
- do they shift to recruiting new bacteria?

In [ ]:
## make sure dates are in chronological order

# 2. reorder MonthYear as a factor in chronological order
sample_nomit$Month_year <- factor(sample_nomit$Month_year,levels = unique(sample_nomit$Month_year))

In [ ]:
# plot sizing
options(repr.plot.width=20, repr.plot.height=15)

# How do the microbiomes of individual colonies change with exposure to stressors like heat and disease?